In [ ]:
# Gerneral stuff
%pip install python-dotenv
%pip install ipywidgets

# ColPali
%pip install "colpali-engine>=0.3.0,<0.4.0"

# GEPA (optimize_anything)
%pip install gepa
%pip install "click==8.3.1" "litellm==1.82.6" "typer==0.24.1" "typer-slim==0.24.0"
%pip install litellm

# UniGPT
%pip install openai

In [1]:
# Von .env Datei einlesen:
## HF_TOKEN
## OPENAI_API_KEY
## OPENAI_API_BASE
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
QUERY_1 = "What are the Scope 1 greenhouse gas emissions?"
QUERY_2 = "Total direct CO2 emissions in metric tons"

In [3]:
# Copy-Paste von ColPali's Repo
# https://huggingface.co/vidore/colpali-v1.3#usage

from typing import cast

import torch
from PIL import Image

from colpali_engine.models import ColPali, ColPaliProcessor

model_name = "vidore/colpali-v1.3"

model = ColPali.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="mps",  # or "mps" if on Apple Silicon, else "cuda:0"
).eval()

processor = ColPaliProcessor.from_pretrained(model_name)

# Your inputs
images = [
     Image.open("./examplePages/0Falsch.png"),
     Image.open("./examplePages/1Falsch.png"),
     Image.open("./examplePages/2Richtig.png"),
     Image.open("./examplePages/3Falsch.png")
]
queries = [QUERY_1, QUERY_2]

# Process the inputs
# batch_images = processor.process_images(images).to(model.device)
batch_queries = processor.process_queries(queries).to(model.device)

# Forward pass
with torch.no_grad():
    image_embeddings_list = []
    for img in images:
        batch = processor.process_images([img]).to(model.device)
        # Labels entfernen - nicht nötig bei Inference
        batch = {k: v for k, v in batch.items() if k != "labels"}
        emb = model(**batch)
        image_embeddings_list.append(emb)
        torch.mps.empty_cache()
    
    image_embeddings = torch.cat(image_embeddings_list, dim=0)
    
    batch_queries = {k: v for k, v in batch_queries.items() if k != "labels"}
    query_embeddings = model(**batch_queries)

scores = processor.score_multi_vector(query_embeddings, image_embeddings)

print("Done.")


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
It is a prefill stage but The `token_type_ids` is not provided. We recommend passing `token_type_ids` to the model to prevent bad attention masking.


Done.


In [4]:
image_names = ["0Falsch.png", "1Falsch.png", "2Richtig.png", "3Falsch.png"]

for i, query in enumerate(queries):
    best_page = scores[i].argmax().item()
    print(f"Query: '{query}'")
    print(f"  → Beste Seite: {image_names[best_page]} (Score: {scores[i][best_page]:.4f})")
    print(f"  → Alle Scores: { {image_names[j]: f'{scores[i][j]:.4f}' for j in range(len(image_names))} }")
    print()

Query: 'What are the Scope 1 greenhouse gas emissions?'
  → Beste Seite: 3Falsch.png (Score: 13.2500)
  → Alle Scores: {'0Falsch.png': '6.3750', '1Falsch.png': '7.3125', '2Richtig.png': '13.0000', '3Falsch.png': '13.2500'}

Query: 'Total direct CO2 emissions in metric tons'
  → Beste Seite: 2Richtig.png (Score: 11.6875)
  → Alle Scores: {'0Falsch.png': '7.2812', '1Falsch.png': '7.8438', '2Richtig.png': '11.6875', '3Falsch.png': '9.3750'}



In [ ]:
import os
import litellm


# models=['mistral-small', 'gemma-3', 'Llama-3.3-70B', 'gemma-3-27b-it', 'gpt-oss-120b', 'Qwen3-Embedding-4B', 'Apertus-8B-Instruct-2509']
# chosenModel = "mistral-small"
chosenModel = "gemma-3-27b-it"
# chosenModel = "gpt-oss-120b"
# chosenModel = "Llama-3.3-70B"
# chosenModel = "Qwen3.5-35B-A3B"

# Kurzer Test-Call an die Uni
try:
    response = litellm.completion(
        model="openai/" + chosenModel, # Oder gemma-3-27b-it
        messages=[{"role": "user", "content": "Test: System aktiv?"}],
        # Die Variablen werden automatisch aus der .env gezogen, 
        # wenn sie OPENAI_API_BASE heißen.
    )
    # 1. Inhalt ausgeben
    print("UniGPT sagt:", response.choices[0].message.content)

    # 2. Token-Usage direkt vom response-Objekt abgreifen
    usage = response.usage
    print(f"\n--- Statistik für {chosenModel} ---")
    print(f"Prompt Tokens:     {usage.prompt_tokens}")
    print(f"Completion Tokens: {usage.completion_tokens}")
    print(f"Gesamt Tokens:     {usage.total_tokens}")

    # 3. Kleiner Bonus: Credit-Berechnung für deine Kosten-Nutzen-Analyse
    # Preise laut deiner UniGPT-Tabelle für Gemma-3-27b-it: 0.09 / 0.16
    cost = (usage.prompt_tokens * 0.09 / 1000) + (usage.completion_tokens * 0.16 / 1000)
    print(f"Geschätzte Kosten: {cost:.4f} Credits")
except Exception as e:
    print("Fehler beim Handshake:", e)

UniGPT sagt: Ja, das System ist aktiv. Ich bin bereit, Ihre Fragen zu beantworten und Ihnen zu helfen. Wie kann ich Ihnen heute assistieren?

--- Statistik für Llama-3.3-70B ---
Prompt Tokens:     40
Completion Tokens: 32
Gesamt Tokens:     72
Geschätzte Kosten: 0.0087 Credits


In [ ]:
# Generated via Claude Sonnet 4.6 based on given Repo-Link
# https://gepa-ai.github.io/gepa/blog/2026/02/18/introducing-optimize-anything/

import gepa.optimize_anything as oa
from colpali_engine.models import ColPali, ColPaliProcessor
from PIL import Image
import torch

# Verkürzt, da Bilder bereits geladen

# def evaluate(query: str) -> float:
#     batch_queries = processor.process_queries([query]).to(model.device)
#     batch_queries = {k: v for k, v in batch_queries.items() if k != "labels"}
    
#     with torch.no_grad():
#         query_embeddings = model(**batch_queries)
    
#     scores = processor.score_multi_vector(query_embeddings, image_embeddings)
    
#     # Score der richtigen Seite (Index 2 = 2Richtig.png)
#     correct_page_score = scores[0][2].item()
#     oa.log(f"Score für richtige Seite: {correct_page_score:.4f}")
#     return correct_page_score


def evaluate(query: str) -> float:
    batch_queries = processor.process_queries([query]).to(model.device)
    batch_queries = {k: v for k, v in batch_queries.items() if k != "labels"}
    
    with torch.no_grad():
        query_embeddings = model(**batch_queries)
    
    # Scores für alle 4 Bilder berechnen
    scores = processor.score_multi_vector(query_embeddings, image_embeddings)
    all_scores = scores[0] # Tensor mit [Score0, Score1, Score2, Score3]
    
    # 1. Score der richtigen Seite (Index 2)
    correct_score = all_scores[2].item()
    
    # 2. Die Scores der falschen Seiten extrahieren (Indices 0, 1, 3)
    # Wir nehmen den höchsten Score der Konkurrenz
    wrong_indices = [0, 1, 3]
    wrong_scores = all_scores[wrong_indices]
    max_wrong_score = torch.max(wrong_scores).item()
    
    # 3. Die entscheidende Metrik: Der Vorsprung (Margin)
    # Ein positiver Wert bedeutet: Die richtige Seite ist auf Platz 1.
    # Ein negativer Wert bedeutet: Eine falsche Seite ist besser.
    margin = correct_score - max_wrong_score
    
    # Logging für die Analyse
    oa.log(f"Correct: {correct_score:.2f} | MaxWrong: {max_wrong_score:.2f} | Margin: {margin:.2f}")
    
    return margin

result = oa.optimize_anything(
    seed_candidate=QUERY_1,
    evaluator=evaluate,
    config=oa.GEPAConfig(
        engine=oa.EngineConfig(max_metric_calls=10),  # klein halten für Tests
        reflection=oa.ReflectionConfig(reflection_lm="openai/gemma-3-27b-it"),
    )
)

print("Beste Query:", result.best_candidate)

Iteration 0: Base program full valset score: -0.25 over 1 / 1 examples
Iteration 1: Selected program 0 score: -0.25
Iteration 1: Proposed new text for current_candidate: What are direct greenhouse gas emissions from owned or controlled sources?
Iteration 1: New subsample score -2.375 is not better than old score -0.25, skipping
Iteration 2: Selected program 0 score: -0.25
Iteration 2: Proposed new text for current_candidate: What are direct greenhouse gas emissions from owned or controlled sources?
Iteration 2: New subsample score -2.375 is not better than old score -0.25, skipping
Iteration 3: Selected program 0 score: -0.25
Iteration 3: Proposed new text for current_candidate: What are direct greenhouse gas emissions from owned or controlled sources?
Iteration 3: New subsample score -2.375 is not better than old score -0.25, skipping
Iteration 4: Selected program 0 score: -0.25
Iteration 4: Proposed new text for current_candidate: What is the definition of Scope 1 greenhouse gas emis

In [19]:
# GEPA-optimierte Query vs. Original-Query

queries = [
    "What are the Scope 1 greenhouse gas emissions?",
    result.best_candidate  # ← direkt von GEPA
]

image_names = ["0Falsch.png", "1Falsch.png", "2Richtig.png", "3Falsch.png"]

batch_queries = processor.process_queries(queries).to(model.device)

with torch.no_grad():
    batch_queries = {k: v for k, v in batch_queries.items() if k != "labels"}
    query_embeddings = model(**batch_queries)

scores = processor.score_multi_vector(query_embeddings, image_embeddings)

In [20]:
labels = ["Original", "GEPA-optimiert"]
for i, (query_label, query_text) in enumerate(zip(labels, queries)):
    best_page = scores[i].argmax().item()
    print(f"[{query_label}]")
    print(f"  → Beste Seite: {image_names[best_page]} (Score: {scores[i][best_page]:.4f})")
    print(f"  → Alle Scores: { {image_names[j]: f'{scores[i][j]:.4f}' for j in range(len(image_names))} }")
    print()

[Original]
  → Beste Seite: 3Falsch.png (Score: 13.2500)
  → Alle Scores: {'0Falsch.png': '6.3750', '1Falsch.png': '7.3125', '2Richtig.png': '13.0000', '3Falsch.png': '13.2500'}

[GEPA-optimiert]
  → Beste Seite: 2Richtig.png (Score: 15.0000)
  → Alle Scores: {'0Falsch.png': '7.9375', '1Falsch.png': '9.1875', '2Richtig.png': '15.0000', '3Falsch.png': '14.9375'}

